# Chapter 04-03 · Splitting I: train, validation, test

**Label:** Core  |  **Time:** ~55 minutes  |  **Difficulty:** moderate

**Prerequisites:** 04-02 for baselines and skill, 03-03 for standard error, 03-02 for sampling.

**Position in the learning path:** module 04, chapter 3 of 8.

---

## Why this matters

The last two chapters compared models on "held-out" data without ever justifying the phrase, and quoted
single numbers - AUC 0.6359, MAE 2.4353 - as though they were properties of the models.

They are not. **They are properties of the model and the particular split**, and this chapter measures
how much of each number was the split. The answer is uncomfortable: the same model, on the same data,
scores anywhere from **0.608 to 0.819** depending on which 30% of members you happen to hold out - and
the number 04-01 and 04-02 reported sits at the **3rd percentile** of that range.

Nothing in those chapters was wrong. Every conclusion they drew was about a gap far larger than this
noise. But a number quoted to four decimals, from one split, is mostly theatre, and this chapter is where
that gets fixed.

## What you will be able to do

- Say why a score on training data is not evidence, and show the gap growing with model complexity
- Measure how much of a reported score is the split rather than the model
- Explain why two splits are not enough as soon as you choose between models, with the number
- Use stratification, and say exactly what it protects
- Choose a test-set size by trading estimate noise against training data, with both quantified
- Read any single-split score as one draw from a distribution

## Warm-up: retrieve, do not reread

1. What is a skill score, and what does a negative one mean?
2. In the module 03 assessment, a model scored *below* the noise floor it was generated with. Why?
3. If a rare class is 13% of the data and you hold out 30% at random, roughly how many positives land in
   the test set out of 157 - and how sure are you?

<br>

*Answers: (1) `(baseline error - model error) / baseline error`; negative means worse than a free rule.
(2) least squares bent the line towards the particular noise in those rows, so error measured on the
fitting data is optimistic. (3) about 21 on average - and the second half of that question is what this
chapter answers.*

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


# SYNTHETIC. The gym panel from 04-01.
def load_members():
    rng = np.random.default_rng(41)
    n = 600
    join_month = rng.integers(1, 13, n)
    commitment = rng.beta(2.0, 2.0, n)
    rows = []
    for member in range(n):
        drifting = 0.0
        base_visits = 2 + 10 * commitment[member]
        for month in range(join_month[member], 25):
            drifting += rng.normal(0.0, 0.35)
            visits = max(0, int(round(rng.poisson(max(0.2, base_visits - drifting)))))
            tickets = int(rng.random() < 0.05 + 0.10 * (visits == 0))
            hazard = 1 / (1 + np.exp(3.0 + 2.5 * commitment[member] - 0.55 * max(0, 4 - visits)))
            cancelled = int(rng.random() < hazard)
            rows.append((member + 1, month, visits, tickets, cancelled))
            if cancelled:
                break
    return pd.DataFrame(rows, columns=["member_id", "month", "visits", "tickets", "cancelled"])


panel = load_members().sort_values(["member_id", "month"])

# the churn task from 04-01: wall at month 12, six-month horizon
CUT, HORIZON = 12, 6
active = panel[(panel.month == CUT) & (panel.cancelled == 0)].member_id.unique()
history = panel[panel.member_id.isin(active) & (panel.month <= CUT)]
future = panel[panel.member_id.isin(active) & (panel.month > CUT)]
left = future[(future.month <= CUT + HORIZON) & (future.cancelled == 1)].member_id.unique()

churn = pd.Series(np.isin(active, left).astype(int), index=active)
features = pd.DataFrame({
    "visits_at_cut": history[history.month == CUT].set_index("member_id").visits.reindex(active),
    "mean_visits_last_3": history[history.month > CUT - 3].groupby("member_id").visits.mean().reindex(active),
    "tenure_months": history.groupby("member_id").size().reindex(active),
    "tickets_so_far": history.groupby("member_id").tickets.sum().reindex(active),
})

# and the regression task from 04-02
panel["next_visits"] = panel.groupby("member_id").visits.shift(-1)
visits_table = panel.dropna(subset=["next_visits"]).copy()
visits_table["mean_so_far"] = (visits_table.groupby("member_id").visits
                               .expanding().mean().reset_index(level=0, drop=True))

print("churn task     : %d members, %d cancel (%.2f%%)"
      % (len(churn), churn.sum(), 100 * churn.mean()))
print("regression task: %d member-months" % len(visits_table))

## Part 1 · Why a training score is not evidence

A decision tree, allowed to grow deeper and deeper on the visits task. Two numbers each time: the error
on the rows it was fitted to, and the error on rows it has never seen.

**Predict before running:** what does the training error do as depth increases? What does the test error
do?

In [ ]:
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor

regression_columns = ["visits", "mean_so_far", "tickets", "month"]
fit_rows, holdout_rows = train_test_split(visits_table, test_size=0.3, random_state=0)

rows = []
for depth in [1, 2, 3, 5, 8, 12, None]:
    tree = DecisionTreeRegressor(max_depth=depth, random_state=0)
    tree.fit(fit_rows[regression_columns], fit_rows.next_visits)
    on_train = mean_absolute_error(fit_rows.next_visits, tree.predict(fit_rows[regression_columns]))
    on_test = mean_absolute_error(holdout_rows.next_visits, tree.predict(holdout_rows[regression_columns]))
    rows.append({"max depth": depth if depth else "unlimited",
                 "MAE on rows it was fitted to": round(on_train, 4),
                 "MAE on unseen rows": round(on_test, 4),
                 "gap": round(on_test - on_train, 4)})
print(pd.DataFrame(rows).to_string(index=False))

**The training error falls all the way to 0.3453 and the test error rises to 3.4772.** An unlimited tree
predicts the rows it was fitted to almost perfectly and is worse than 04-02's *global mean* baseline
(2.9876) on rows it has not seen.

Two things worth naming precisely.

**The training error is not a bad estimate of the test error - it is not an estimate of it at all.** It
is a measure of how well the model can reproduce data it has already seen, and it can be driven to zero
by anyone willing to add capacity. A model with one leaf per row scores perfectly and knows nothing.

**The best depth is 3, and you can only see that in the second column.** The tree at depth 3 scores
2.4535, at depth 12 it scores 2.8312, and the training error prefers 12 by a distance. Any procedure that
chooses using the training score chooses the worst model available.

This is why held-out data exists, and it is the entire justification for everything that follows.

In [ ]:
depths = [1, 2, 3, 5, 8, 12, 16, 20]
train_curve, test_curve = [], []
for depth in depths:
    tree = DecisionTreeRegressor(max_depth=depth, random_state=0)
    tree.fit(fit_rows[regression_columns], fit_rows.next_visits)
    train_curve.append(mean_absolute_error(fit_rows.next_visits, tree.predict(fit_rows[regression_columns])))
    test_curve.append(mean_absolute_error(holdout_rows.next_visits, tree.predict(holdout_rows[regression_columns])))

fig, ax = plt.subplots(figsize=(8, 4.6))
ax.plot(depths, train_curve, "o-", color="#0072B2", label="rows it was fitted to")
ax.plot(depths, test_curve, "s-", color="#D55E00", label="unseen rows")
best = int(np.argmin(test_curve))
ax.plot(depths[best], test_curve[best], "*", color="#000000", markersize=18)
ax.annotate("best depth: %d" % depths[best], (depths[best], test_curve[best]),
            textcoords="offset points", xytext=(10, 10))
ax.axhline(2.9876, color="#999999", linestyle=":", label="global mean baseline (04-02)")
ax.set_xlabel("maximum tree depth")
ax.set_ylabel("mean absolute error")
ax.set_title("The two curves go opposite ways. Only one of them is evidence")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

The scissors shape is the single most important picture in applied machine learning, and it will return
in 05-08 with a proper treatment. What matters here is the workflow consequence: **the blue curve is
always available and always misleading, and the orange one requires you to have set data aside before
you started.**

## Part 2 · The split is a lottery

So a held-out score is the evidence. How much can you trust one?

The honest way to find out is to hold out a different 30% and see what changes. Nothing about the model
or the data is altered - only which members happen to be on which side.

**Predict before running:** 04-01 and 04-02 both reported AUC 0.6359 for this model. Over 200 different
random splits, how wide will the range be?

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score


def fit_and_score(seed, test_size=0.3, stratify=True):
    train_X, test_X, train_y, test_y = train_test_split(
        features, churn, test_size=test_size, random_state=seed,
        stratify=churn if stratify else None)
    centre, spread = train_X.mean(), train_X.std()
    model = LogisticRegression(max_iter=2000).fit((train_X - centre) / spread, train_y)
    return roc_auc_score(test_y, model.predict_proba((test_X - centre) / spread)[:, 1])


scores = np.array([fit_and_score(seed) for seed in range(200)])
reported = fit_and_score(0)

print("the same model and the same data, 200 different splits:")
print("  mean   %.4f" % scores.mean())
print("  sd     %.4f" % scores.std())
print("  range  %.4f to %.4f" % (scores.min(), scores.max()))
print("  middle 90%%  %.4f to %.4f" % (np.percentile(scores, 5), np.percentile(scores, 95)))
print()
print("  the number 04-01 and 04-02 reported (seed 0): %.4f" % reported)
print("  only %.0f%% of the 200 splits scored lower" % (100 * np.mean(scores < reported)))

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.4))
ax.hist(scores, bins=28, color="#cfe3f3", edgecolor="#0072B2")
ax.axvline(reported, color="#D55E00", linewidth=2.5)
ax.text(reported + 0.004, ax.get_ylim()[1] * 0.9, "the number two chapters\nreported: %.4f" % reported,
        color="#D55E00", fontsize=9)
ax.axvline(scores.mean(), color="#000000", linestyle="--", linewidth=1.5)
ax.text(scores.mean() + 0.004, ax.get_ylim()[1] * 0.55, "mean of 200 splits\n%.4f" % scores.mean(),
        fontsize=9)
ax.set_xlabel("AUC on the held-out 30%")
ax.set_ylabel("number of splits")
ax.set_title("One model, one dataset, 200 answers")
plt.tight_layout()
plt.show()

**0.6080 to 0.8185, with a standard deviation of 0.0425.** The number reported twice in this course sits
at the **3rd percentile** - it was an unlucky draw, and every conclusion drawn from it happened to be
conservative rather than flattering.

Three things follow, and the third is the one people resist.

**A single split's score has a standard error, and it is large.** 0.0425 here, on 157 held-out members
containing 21 cancellers. Quoting such a number to four decimal places implies a precision that does not
exist. 03-03 said this about any quantity computed from a sample; a test score is one.

**Two models differing by less than that spread have not been distinguished.** 04-02 compared a logistic
regression at 0.6359 against a one-rule threshold at 0.6311 and said the gap "is not measurable". This is
the measurement that justifies the claim.

**And you cannot fix it by trying several splits and keeping the best.** That is the same error as
choosing a model on the test set, arriving from a different direction - which is the subject of Part 4.
The fix is to average over many splits, which is cross-validation, and it is 04-07.

## Part 3 · Stratification

There is one part of the lottery you can simply switch off.

A random split takes 30% of the *rows*, and says nothing about the 13.44% of them that are the interesting
class. On a small or imbalanced dataset that leaves the test set's base rate to chance.

In [ ]:
plain_rates, stratified_rates = [], []
for seed in range(2000):
    _, _, _, plain_y = train_test_split(features, churn, test_size=0.3, random_state=seed)
    _, _, _, stratified_y = train_test_split(features, churn, test_size=0.3,
                                             random_state=seed, stratify=churn)
    plain_rates.append(plain_y.mean())
    stratified_rates.append(stratified_y.mean())
plain_rates, stratified_rates = np.array(plain_rates), np.array(stratified_rates)

held_out_size = len(plain_y)
print("base rate of the held-out set, over 2000 splits")
print("  %-12s mean %.4f  sd %.4f  range %.4f to %.4f"
      % ("plain", plain_rates.mean(), plain_rates.std(), plain_rates.min(), plain_rates.max()))
print("  %-12s mean %.4f  sd %.4f  range %.4f to %.4f"
      % ("stratified", stratified_rates.mean(), stratified_rates.std(),
         stratified_rates.min(), stratified_rates.max()))
print()
print("  in counts: a plain split puts between %d and %d cancellers in the test set of %d"
      % (round(plain_rates.min() * held_out_size), round(plain_rates.max() * held_out_size), held_out_size))
print("  %.1f%% of plain splits are more than 3 percentage points from the true rate of %.2f%%"
      % (100 * np.mean(np.abs(plain_rates - churn.mean()) > 0.03), 100 * churn.mean()))

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.4))
ax.hist(100 * plain_rates, bins=30, color="#f6d3bd", edgecolor="#D55E00", label="plain random split")
ax.axvline(100 * stratified_rates[0], color="#0072B2", linewidth=3,
           label="stratified (every split, exactly)")
ax.axvline(100 * churn.mean(), color="#000000", linestyle="--", linewidth=1.2,
           label="true base rate")
ax.set_xlabel("percentage of the held-out set that cancels")
ax.set_ylabel("number of splits")
ax.set_title("Stratification removes one source of the lottery entirely")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

**A plain split puts between 11 and 35 cancellers in the test set** - a three-fold range - and one split
in five is more than three percentage points off the true rate. **Stratified, it is 13.38% every single
time**, with a standard deviation of exactly zero.

That is worth being precise about, because stratification is often over-sold:

- **What it fixes:** the composition of the split. The test set now has the same class balance as the
  data, so the base rate a metric is measured against is fixed rather than random, and comparisons across
  experiments are on the same footing.
- **What it does not fix:** *which* members are in the test set. The 200-split spread in Part 2 was
  already stratified, and it still ran from 0.608 to 0.819. Stratification removes one component of the
  noise and leaves the larger one.
- **When it matters most:** small datasets, rare classes, and any grouped column you care about - you can
  stratify on a region, a product line, or a source, not only on the target.
- **When it does nothing:** large, balanced datasets, where the law of large numbers has already done it
  for you.

**The practical rule is simply to always use it for classification.** It costs one keyword, it never
hurts, and the case where it saves you - a rare class and a small dataset - is exactly the case where you
are least able to notice the problem it prevents.

## Part 4 · Why two splits are not enough

Here is the argument that most people accept in principle and violate in practice.

You have a training set and a test set. You try a model, score it on the test set, adjust something, score
it again, try a third idea, score that. Every score is on held-out data, so this feels safe.

It is not. **The moment you use the test score to choose, the test set is participating in fitting** -
not through gradients, but through you. And the more options you try, the more the winner's score is
inflated by having got lucky on those particular rows.

The demonstration: forty candidate models, each a logistic regression on four columns drawn at random
from the four real features plus **twenty columns of pure noise**. Most are worthless by construction.

**Predict before running:** the best of forty will score well on whatever set chose it. How much of that
score is real?

In [ ]:
noise_rng = np.random.default_rng(7)
wide = features.copy()
for j in range(20):
    wide["noise_%d" % j] = noise_rng.normal(size=len(features))

# a proper three-way split: 60% train, 20% validation, 20% test
train_X, rest_X, train_y, rest_y = train_test_split(
    wide, churn, test_size=0.4, random_state=0, stratify=churn)
val_X, test_X, val_y, test_y = train_test_split(
    rest_X, rest_y, test_size=0.5, random_state=0, stratify=rest_y)
print("train %d, validation %d, test %d" % (len(train_X), len(val_X), len(test_X)))


def try_candidate(chosen):
    centre, spread = train_X[chosen].mean(), train_X[chosen].std()
    model = LogisticRegression(max_iter=2000).fit((train_X[chosen] - centre) / spread, train_y)
    return (roc_auc_score(val_y, model.predict_proba((val_X[chosen] - centre) / spread)[:, 1]),
            roc_auc_score(test_y, model.predict_proba((test_X[chosen] - centre) / spread)[:, 1]))


candidates = []
for _ in range(40):
    chosen = list(noise_rng.choice(wide.columns, size=4, replace=False))
    on_val, on_test = try_candidate(chosen)
    candidates.append({"columns": chosen, "validation AUC": on_val, "test AUC": on_test})
candidates = pd.DataFrame(candidates)

winner = candidates.loc[candidates["validation AUC"].idxmax()]
print()
print("average candidate      : validation %.4f, test %.4f"
      % (candidates["validation AUC"].mean(), candidates["test AUC"].mean()))
print("best on validation     : validation %.4f, test %.4f   <- the honest report"
      % (winner["validation AUC"], winner["test AUC"]))
print("the selection premium  : %.4f of AUC, evaporated"
      % (winner["validation AUC"] - winner["test AUC"]))
print()
print("best on TEST           : test %.4f   <- what you would have reported"
      % candidates["test AUC"].max())
print("                          had you been choosing on the test set")

**The winner scores 0.7532 on the set that selected it and 0.6366 on a set that did not.** The gap of
**0.1166** is not model degradation - it is the part of the validation score that was luck, and it
vanishes the moment the model meets rows that had no say in choosing it.

Remember what the average candidate scores: **0.5411 on validation**. These are mostly random noise
columns. Selection alone lifted the *reported* score of a mostly-worthless model by more than 0.2.

Watch it grow with the number of things you try.

In [ ]:
# One run of 40 is a single noisy realisation. Build a bigger pool and average
# over many draws of size k, so the shape is the effect rather than the luck.
pool = np.array([try_candidate(list(noise_rng.choice(wide.columns, size=4, replace=False)))
                 for _ in range(300)])

draw_rng = np.random.default_rng(3)
rows = []
for k in [1, 2, 5, 10, 20, 40, 80]:
    best_validation, its_test = [], []
    for _ in range(200):
        drawn = pool[draw_rng.choice(len(pool), size=k, replace=False)]
        winner_of_draw = drawn[np.argmax(drawn[:, 0])]
        best_validation.append(winner_of_draw[0])
        its_test.append(winner_of_draw[1])
    rows.append({"candidates tried": k,
                 "best validation AUC": round(float(np.mean(best_validation)), 4),
                 "its test AUC": round(float(np.mean(its_test)), 4),
                 "premium": round(float(np.mean(best_validation) - np.mean(its_test)), 4)})
growth = pd.DataFrame(rows)
print("averaged over 200 draws at each size:")
print(growth.to_string(index=False))

fig, (left, right) = plt.subplots(1, 2, figsize=(12.5, 4.4))
left.scatter(candidates["validation AUC"], candidates["test AUC"], s=28,
             color="#999999", label="the 40 candidates")
left.scatter([winner["validation AUC"]], [winner["test AUC"]], s=140, marker="*",
             color="#D55E00", label="chosen on validation")
limits = [0.3, 0.85]
left.plot(limits, limits, color="#000000", linewidth=0.8, linestyle=":")
left.set_xlabel("AUC on validation (the set that chose)")
left.set_ylabel("AUC on test (the set that did not)")
left.set_title("The winner is far above the line, not on it")
left.legend(fontsize=8)

right.plot(growth["candidates tried"], growth["best validation AUC"], "o-",
           color="#D55E00", label="best validation score")
right.plot(growth["candidates tried"], growth["its test AUC"], "s-",
           color="#0072B2", label="what it really scores")
right.set_xscale("log")
right.set_xlabel("number of candidates tried (log scale)")
right.set_ylabel("AUC")
right.set_title("The more you try, the more the winner's score is luck")
right.legend(fontsize=8)

plt.tight_layout()
plt.show()

The left panel is the mechanism in one picture. If validation and test measured the same thing, the points
would scatter around the dotted diagonal - and they do, **except the one that was chosen for being high**.
Selecting the maximum of forty noisy numbers selects the noise along with the signal.

The right panel is why the problem is not academic. **The two lines separate and never come back
together.** With one candidate there is nothing to select, and the premium is -0.0121 - zero, within
noise. By ten it is 0.0779, by forty 0.1137, by eighty 0.1199.

Read the middle column instead of the first, though, because that is the finding: **the winner's actual
test score stops improving after about ten candidates** - 0.6342 at ten, 0.6223 at eighty, drifting
slightly *down* - while the score you would *report* keeps climbing from 0.7121 to 0.7422. Past ten
candidates, every additional thing you try buys **no real performance at all** and simply inflates the
number you would put in a slide.

**A modern hyperparameter search tries hundreds or thousands.**

In [ ]:
fig, ax = plt.subplots(figsize=(11, 2.9))
segments = [("training  60%", 0.0, 0.6, "#0072B2", "the model reads it\nas often as it likes"),
            ("validation  20%", 0.6, 0.2, "#e8a33d", "you read it as often\nas you like"),
            ("test  20%", 0.8, 0.2, "#D55E00", "read ONCE,\nat the very end")]
for label, start, width, colour, note in segments:
    ax.add_patch(plt.Rectangle((start, 0.35), width, 0.5, facecolor=colour, edgecolor="white",
                               linewidth=3))
    ax.text(start + width / 2, 0.6, label, ha="center", va="center", color="white",
            fontsize=11, fontweight="bold")
    ax.text(start + width / 2, 0.16, note, ha="center", va="center", fontsize=8.5, color="#444444")
ax.text(0.5, 0.96, "one dataset, three jobs, three viewing budgets", ha="center", fontsize=11)
ax.set_xlim(-0.01, 1.01)
ax.set_ylim(0, 1.1)
ax.set_xticks([])
ax.set_yticks([])
for side in ax.spines.values():
    side.set_visible(False)
plt.tight_layout()
plt.show()

### So: three sets, and one rule each

| Set | Used for | Rule |
|---|---|---|
| **Training** | fitting parameters | the model sees it as often as it likes |
| **Validation** | choosing between models, features, hyperparameters | you see it as often as you like; the score is optimistic by construction |
| **Test** | estimating what the chosen model will do in production | **looked at once, at the end, after everything is decided** |

The test set's whole value is that nothing about the model was chosen using it. **Every look costs some
of that**, and the cost is invisible - there is no warning, only a number that is quietly too high.

A useful way to hold it: the validation score answers "which of these is best?" and the test score answers
"how good is the one I picked?". They are different questions, and the second cannot be answered by the
set that answered the first.

**In practice**, on data this size, you would not carve out a fixed validation set at all - 104 members is
far too few to choose between forty models reliably, as the 0.1166 premium shows. You would use
cross-validation for the choosing and keep a single untouched test set for the final number. That is
04-07. The three-way split is the idea; cross-validation is the efficient implementation of it.

## Part 5 · How big should the test set be?

A trade-off with two sides that pull opposite ways, and both are measurable.

- A **larger** test set gives a more precise estimate of performance.
- A **smaller** test set leaves more rows to train on, so the model is better.

**Predict before running:** which effect is bigger here?

In [ ]:
rows = []
for fraction in [0.1, 0.2, 0.3, 0.5]:
    repeated = np.array([fit_and_score(seed, test_size=fraction) for seed in range(200)])
    rows.append({"test share": "%.0f%%" % (100 * fraction),
                 "test rows": int(fraction * len(churn)),
                 "training rows": len(churn) - int(fraction * len(churn)),
                 "mean AUC": round(repeated.mean(), 4),
                 "sd across splits": round(repeated.std(), 4)})
sizes_table = pd.DataFrame(rows)
print(sizes_table.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.4))
shares = [0.1, 0.2, 0.3, 0.5]
means = sizes_table["mean AUC"].to_numpy()
spreads = sizes_table["sd across splits"].to_numpy()
ax.errorbar([100 * f for f in shares], means, yerr=spreads, fmt="o-", color="#0072B2",
            capsize=6, linewidth=2, markersize=8)
for share, mean_value, spread in zip(shares, means, spreads):
    ax.annotate("sd %.4f" % spread, (100 * share, mean_value + spread),
                textcoords="offset points", xytext=(8, 4), fontsize=8.5, color="#D55E00")
ax.set_xlabel("share of the data held out for testing (%)")
ax.set_ylabel("AUC, averaged over 200 splits")
ax.set_title("The bars shrink and the dot barely moves", fontsize=11)
plt.tight_layout()
plt.show()

**The noise falls by a factor of three - 0.0857 to 0.0280 - and the mean barely moves**, from 0.7292 to
0.7203.

So on this dataset the precision side dominates completely: going from a 10% to a 50% test set costs
about 0.009 of average AUC and buys a three-fold reduction in how much your reported number depends on
luck. The standard error is following 03-02's square-root law - five times the test rows, a bit over twice
the precision.

That said, do not over-fit the lesson to this dataset. The right answer depends on where you are:

| Situation | Typical choice | Why |
|---|---|---|
| A few hundred rows, as here | cross-validation, not a fixed split | any single test set is too small to be precise, whatever fraction you choose |
| Thousands to tens of thousands | 20-30% test | the usual default, and the trade-off is mild in this range |
| Millions | a fixed number of rows, not a fraction | 10,000 test rows is plenty; 20% of ten million is wasted training data |
| Rare positive class | size the test set by **positives**, not rows | 21 cancellers is the real sample size here, not 157 members |

That last row is the one worth carrying. **The precision of a classification metric is governed by the
count of the rare class**, so a test set with a million rows and eleven positives is a small test set. It
is the same point 04-01's E10 arrived at from the other direction, and it is why the sizing question is
answered in events rather than rows.

## Common misconceptions

**"My model got 0.85 on the test set, so it will get 0.85 in production."**
It got 0.85 on one sample of held-out rows. The same model on this data ranged over 0.21 of AUC depending
on the split. A test score is an estimate with a standard error, like any other number computed from a
sample.

**"I only looked at the test set a few times."**
Forty looks cost 0.1166 of AUC here. A few looks cost less and not nothing, and there is no counter
telling you how much you have spent.

**"Stratification makes the split fair."**
It fixes the class balance and nothing else. The stratified 200-split spread was still 0.608 to 0.819.

**"A bigger test set is always better."**
It is better for precision and worse for training. On small data the honest answer is not a bigger test
set but cross-validation, which uses every row for both.

**"Random splitting is the default, so it must be safe."**
It is safe when rows are independent. The next chapter is entirely about the very common case where they
are not - and this chapter's own data is one, since a member-month table has the same member many times.

**"If the test score is much worse than validation, the model degraded."**
Nothing degraded. The validation score was inflated by having been used to choose.

**"I will hold out a test set at the end."**
Then it is not held out. Rows set aside after you have looked at everything have already informed your
choices.

## Exercises

Solutions: `solutions/04_workflow/04-03_splitting_basics_solutions.ipynb`.

### Quick understanding

**E1.** State in one sentence each what the training, validation and test sets are for, and how many times
each may be looked at.

**E2.** Why is a training score not a bad estimate of generalisation, but no estimate at all?

**E3.** What exactly does stratification fix, and what does it leave untouched? Give the two numbers from
this chapter that show both halves.

### Hand calculation

**E4.** A dataset has 400 rows, 24 of them positive. You hold out 25% at random. Compute the expected
number of positives in the test set. Then, given that the standard deviation of that count is about 2.1,
give a range you would not be surprised by, and say what that implies about a metric computed on it.

**E5.** A test set has 150 rows and your model gets 120 right. The standard error of a proportion is
`sqrt(p(1-p)/n)`. Compute the accuracy and its standard error, then state the range within about two
standard errors. Now do the same for a test set of 1,500 rows at the same accuracy. How many times more
data was needed to halve the interval?

**E6.** You try 20 models and pick the best on the test set. Each model's test score is the true score
plus independent noise with a standard deviation of 0.04. Roughly how much higher than the true value do
you expect the winner's *reported* score to be? (The expected maximum of 20 standard normal draws is about
1.87.) State the answer in AUC points.

**E7.** Using this chapter's table of test-set sizes, compute how many test rows would be needed to bring
the standard deviation of the AUC estimate down to 0.01, assuming it follows the square-root law from the
30% row. Comment on whether that is achievable here.

### Coding

**E8.** Write `split_spread(model_builder, n_splits=200)` returning the mean, standard deviation and 5th
and 95th percentiles of a model's held-out score across many random splits. Run it for the logistic
regression and for 04-02's one-rule threshold, and say whether the two are distinguishable.

**E9.** Repeat Part 2's experiment without stratification. Does the spread of the AUC get wider? Report
both standard deviations and explain the size of the difference.

**E10.** Reproduce the depth curve from Part 1 five times with five different random splits, and plot all
five test curves on one axis. Does the best depth stay at 3? What does your answer say about choosing a
hyperparameter from a single split?

**E11.** Write `honest_search(candidates, train, validation, test)` that selects on validation and reports
both the validation and the test score of the winner, and prints a warning when the gap exceeds a
threshold you choose. Justify the threshold.

**E12.** Increase the number of noise columns in Part 4 from 20 to 100 and re-run with 200 candidates.
Report the winner's validation and test scores. Does the premium grow, and does the winner's test score
improve at all?

### Interpretation

**E13.** A paper reports that a new method beats the previous best by 0.4% accuracy on a benchmark with a
2,000-row test set. Say what you would want to know before believing it, and compute roughly what the
standard error on that test set is at 90% accuracy.

**E14.** A colleague's model scores 0.91 AUC on validation and 0.88 on test, and they want to report 0.91
because "the test set is smaller and noisier". Give your response in three sentences.

### Debugging

**E15.** Your test score is *higher* than your validation score, consistently, across several models. Give
two explanations and the check that distinguishes them.

**E16.** A model scores 0.99 AUC on the test set and 0.72 on next month's live data. The split was
random, stratified and never inspected. Name the most likely cause and say why this chapter's discipline
did not prevent it.

### Exam and interview reasoning

**E17.** "Why do you need a validation set if you already have a test set?" Answer in under a minute with
a concrete number. Then answer: "we only have 400 rows - what do you do instead?"

### Transfer to a different situation

**E18.** You are given 50,000 labelled images from 300 patients for a diagnostic model. Describe how you
would split them and what you would stratify on. Name the mistake that a plain random split of the 50,000
images would cause, and how you would detect it if somebody had already made it.

### Explain it to someone non-technical

**E19.** Explain to a manager, in under 90 words, why you cannot report the best number you have seen
during development, using an analogy that is not about machine learning.

### Optional challenge

**E20.** Build the **selection-premium curve** properly: for `k` from 1 to 200, generate `k` random
candidate models, select the best on validation, and record the gap between its validation and test
scores - averaged over 20 repetitions so the curve is readable. Plot the premium against `k` on a log
axis. What function of `k` does it look like, and what does that imply about a hyperparameter search with
1,000 configurations?

In [ ]:
# Your workspace. In memory: panel, churn, features, visits_table, regression_columns,
# fit_rows, holdout_rows, fit_and_score, scores, wide, train_X, val_X, test_X,
# train_y, val_y, test_y, try_candidate, candidates.

## Mastery check

- [ ] Explain why a training score can be driven to zero and what that proves
- [ ] Read the scissors picture and say which curve chooses the model
- [ ] State that a single test score has a standard error, and estimate it by re-splitting
- [ ] Say what stratification fixes and what it does not, with numbers for both
- [ ] Explain the selection premium and why it grows with the number of things tried
- [ ] Assign each of the three sets its purpose and its viewing budget
- [ ] Size a test set in events rather than rows

## What should now feel instinctive

- Splitting before looking at anything, and writing the split into the first cell
- Passing `stratify=` on every classification split without thinking about it
- Reading any single held-out score as one draw, and re-splitting to see the spread
- Refusing to quote a test number after using the test set to choose
- Asking "how many positives are in the test set?" before "how many rows?"

## Flashcards

| Front | Back |
|---|---|
| Why hold out data | A training score measures reproduction, not generalisation; it goes to 0.3453 while the truth is 3.4772 |
| The scissors | Training error falls with capacity, test error falls then rises. Best depth 3 here |
| The split lottery | Same model, 200 splits: AUC 0.608 to 0.819, sd 0.0425 |
| Stratification fixes | The class balance of each split - exactly 13.38% every time here |
| Stratification does not fix | Which rows land where; the stratified spread was still 0.21 wide |
| Validation set | Chooses between models. Its score is optimistic by construction |
| Test set | Estimates the chosen model's performance. Looked at once, at the end |
| Selection premium | Best-of-40 scored 0.7532 on validation, 0.6366 on test - a gap of 0.1166 |
| Why the premium grows | Selecting the maximum of many noisy scores selects the noise too |
| Test-set size trade-off | 10% to 50% here: sd falls 0.0857 to 0.0280, mean AUC falls 0.0089 |
| Sizing rule for rare classes | Count the positives, not the rows. 21 cancellers, not 157 members |
| The efficient fix on small data | Cross-validation for choosing, one untouched test set for reporting (04-07) |

## Next

**04-04 · Splitting II: grouped and chronological.** Everything in this chapter assumed one thing that is
false for most real datasets, and false for the very table it used: **that rows are independent.**

04-01's E18 already built the member-month version of the churn problem - 6,318 rows in which the average
member appears about eleven times. A random split of those rows puts member 314's month 7 in training and
their month 8 in test, and the "held-out" score then measures whether the model can recognise a member it
has already met. The next chapter measures how large that inflation is, and fixes it - along with the
same problem in its temporal form, where the future leaks into the past.